# 006 - SQL Runner for Notebook · 데모

Jupyter 셀 안에서 SQL 을 작성하고 **▶ 실행 버튼으로 Python 콜백을 호출** 해 결과를 즉시 확인하는 single-file 위젯.

## 005 와의 차이

- **005** = HTML/JS only, inline popup autocomplete · 커서 위치 정밀 인서트 가능 · 단 Python 콜백 호출 불가
- **006 (이 데모)** = ipywidgets 기반, ▶ 실행 버튼이 `on_execute(sql)` 호출 · 라이브 syntax 강조 프리뷰 · DataFrame 결과 자동 표 렌더

## 시도할 것

- 좌측 트리에서 컬럼 클릭 → 쿼리에 자동 삽입
- 추천 칩 (`💡 추천`) 클릭 → 쿼리에 자동 삽입
- Textarea 에서 자유롭게 편집 (Enter = 줄바꿈 자연스럽게)
- 라이브 미리보기 영역에 키워드/문자열/숫자/주석 컬러 강조
- ▶ 실행 → 아래 Output 영역에 DataFrame 표 표시


---

## 1️⃣ 셋업: 임시 SQLite DB + pandas

실행 콜백으로 `pd.read_sql(sql, conn)` 을 주입.


In [ ]:
import os, sys, sqlite3, tempfile
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from sql_runner import SQLRunner

# 임시 DB 생성
db_path = os.path.join(tempfile.gettempdir(), "sql_runner_demo.db")
# ⚠️ ipywidgets 버튼 콜백은 다른 스레드에서 실행되므로 SQLite 의 thread
# check 비활성화. 데모(단일 사용자) 용도라 안전. 멀티스레드 production 은
# 매 호출마다 sqlite3.connect 하는 패턴(아래 주석 참고) 권장.
conn = sqlite3.connect(db_path, check_same_thread=False)
conn.executescript('''
DROP TABLE IF EXISTS users;
DROP TABLE IF EXISTS orders;
CREATE TABLE users (
    id INTEGER PRIMARY KEY, name TEXT NOT NULL, region TEXT,
    signup_at TEXT
);
CREATE TABLE orders (
    id INTEGER PRIMARY KEY, user_id INTEGER, amount REAL,
    status TEXT, ordered_at TEXT,
    FOREIGN KEY(user_id) REFERENCES users(id)
);
INSERT INTO users VALUES
    (1, '김알리스', '서울', '2024-01-15'),
    (2, '이밥', '부산', '2024-02-20'),
    (3, '박찰리', '대구', '2024-03-10'),
    (4, '최다나', '서울', '2024-04-01');
INSERT INTO orders VALUES
    (1, 1, 39000, 'paid', '2024-05-01'),
    (2, 1, 12500, 'paid', '2024-05-08'),
    (3, 2, 88000, 'paid', '2024-05-12'),
    (4, 3, 15000, 'cancelled', '2024-05-15'),
    (5, 1, 22000, 'paid', '2024-06-02'),
    (6, 4, 99000, 'paid', '2024-06-10'),
    (7, 2, 33000, 'pending', '2024-06-15');
''')
conn.commit()
print(f'✓ DB 준비 완료: {db_path}')


---

## 2️⃣ SQLRunner 위젯 띄우기

`on_execute=lambda sql: pd.read_sql(sql, conn)` 으로 콜백 주입.


In [ ]:
# 가장 간단: lambda + 공유 conn (위 셋업에서 check_same_thread=False)
runner = SQLRunner(on_execute=lambda sql: pd.read_sql(sql, conn))

# 더 안전한 패턴 — 매 호출마다 connect 하면 thread 걱정 0:
#   def run_sql(sql, path=db_path):
#       with sqlite3.connect(path) as c:
#           return pd.read_sql(sql, c)
#   runner = SQLRunner(on_execute=run_sql)
#
# 또는 편의 메서드 — sqlite + pandas 자동 thread-safe wrap:
#   runner = SQLRunner.with_sqlite(db_path)
runner.from_sqlite(db_path)
runner.set_query(
    "-- 사용자별 결제 합계 (status='paid' 만)\n"
    "SELECT u.name, u.region, SUM(o.amount) AS total\n"
    "FROM users u\n"
    "JOIN orders o ON o.user_id = u.id\n"
    "WHERE o.status = 'paid'\n"
    "GROUP BY u.name, u.region\n"
    "ORDER BY total DESC;"
)
runner.show()


**▶ 실행 버튼을 눌러보세요!** 위 쿼리가 그대로 `pd.read_sql` 로 전달되어 DataFrame 결과가 표로 표시됩니다.

에디터에 추가 시도:
- 좌측 `users` 컬럼 클릭 → 쿼리에 컬럼명 자동 삽입
- 추천 칩 클릭 → 컨텍스트에 맞는 키워드/컬럼 즉시 삽입
- 자유롭게 쿼리 수정 후 ▶ 실행 다시 누르기


---

## 3️⃣ pandas DataFrame 으로 직접 등록 (사내 분석 노트북 패턴)

이미 메모리에 로드된 DataFrame 들을 SQL 처럼 다루는 시나리오. `on_execute` 가 `pandasql` / `duckdb` / 사내 엔진 등을 호출하면 됩니다. 데모에선 단순히 SQL 문자열을 그대로 출력만 합니다.


In [ ]:
df_events = pd.DataFrame({
    'event_id': range(1, 11),
    'user_id': [1, 1, 2, 3, 1, 4, 2, 4, 3, 1],
    'event_type': ['click', 'view', 'click', 'purchase', 'view',
                   'click', 'view', 'purchase', 'view', 'click'],
    'value': [None, None, None, 39000, None,
              None, None, 55000, None, None],
})

def echo_only(sql):
    print('=== 받은 SQL (echo) ===')
    print(sql)
    print('=== 실제 실행 엔진과 연결되면 결과가 여기에 나옵니다 ===')

echo_runner = SQLRunner(on_execute=echo_only)
echo_runner.from_dataframes({'events': df_events})
echo_runner.set_query('SELECT event_type, COUNT(*) FROM events GROUP BY event_type;')
echo_runner.show()


---

## 4️⃣ 콜백 없이 위젯만 띄우는 경우

`on_execute` 미주입 시 ▶ 실행 클릭 → SQL 텍스트만 표시 (시연/디자인 확인용).


In [ ]:
preview_only = SQLRunner()   # on_execute=None
preview_only.from_dict({
    'customers': ['id', 'name', 'phone'],
    'invoices':  [('id', 'INT'), ('customer_id', 'INT'), ('total', 'REAL')],
})
preview_only.set_query('SELECT * FROM customers;')
preview_only.show()


---

## 5️⃣ 컨텍스트 추천 / 구문 강조 — 단위 함수 직접 호출

위젯 없이 `highlight_sql_html` / `detect_context` / `get_suggestions` 만 사용해 외부 시스템에 통합하고 싶을 때.


In [ ]:
from sql_runner import highlight_sql_html, detect_context, get_suggestions
from IPython.display import HTML, display

queries = [
    'SELECT ',
    'SELECT * FROM ',
    'SELECT * FROM orders WHERE ',
    'SELECT * FROM orders WHERE orders.',
    "SELECT name, SUM(amount) FROM users JOIN orders ON o.user_id = u.id WHERE status = 'paid' GROUP BY ",
]
schema = {
    'users': [{'name': 'id', 'type': 'INT', 'doc': ''},
              {'name': 'name', 'type': 'TEXT', 'doc': ''}],
    'orders': [{'name': 'id', 'type': 'INT', 'doc': ''},
               {'name': 'user_id', 'type': 'INT', 'doc': ''},
               {'name': 'amount', 'type': 'REAL', 'doc': ''},
               {'name': 'status', 'type': 'TEXT', 'doc': ''}],
}
for q in queries:
    ctx = detect_context(q)
    sugs = get_suggestions(q, schema)
    top5 = ', '.join(s['label'] for s in sugs[:5])
    print(f'  {q!r:80}  →  ctx={ctx:18}  top: {top5}')


---

## 정리

- **▶ 실행** 으로 사용자 콜백 호출 가능 → DataFrame 결과 자동 표 렌더
- 라이브 SQL 구문 강조 프리뷰
- 컨텍스트 인식 추천 + 클릭 가능 트리/칩
- 005 와 동일한 스키마 등록 헬퍼 (`add_table` / `from_dict` / `from_sqlite` / `from_dataframes`)
- inline popup autocomplete 가 필요하면 005 사용 권장
